<a href="https://colab.research.google.com/github.com/denisejroth/bags-vectors-transformers/blob/main/day1/notebooks/1_intro_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 1 — Working with Text Data in Python  ·  **SOLUTIONS**

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version of the Day 1 notebook. Every **✏️ Exercise** is filled
in with one possible answer. There is usually more than one correct way to solve each one —
if your approach differs but gives the right result, that is completely fine.


## 0. Setup

In [ ]:
# Core libraries
import re
import string
from collections import Counter

import matplotlib.pyplot as plt

# NLTK for text preprocessing
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Download the NLTK data we need (safe to run more than once)
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

print("Setup complete!")

## 1. Text is just data

In [ ]:
response = "I really love this course! The examples are great and super helpful :)"

print(response)
print()
print("Type:", type(response))
print("Length (characters):", len(response))

In [ ]:
# Make everything lowercase
print(response.lower())

# Split into words on whitespace
print(response.split())

# Count how many words (roughly)
print("Rough word count:", len(response.split()))

> **✏️ Exercise 1**
>
> Create your own string variable called `my_text` containing a sentence about your
> research. Then print it in **uppercase** and print how many characters it has.


In [ ]:
# ✅ Solution
my_text = "My research looks at how incivility spreads in online political discussions."

print(my_text.upper())
print("Characters:", len(my_text))

## 2. From one document to a corpus

In [ ]:
corpus = [
    "I love the new bus policy, it makes my commute so much easier!",
    "The new bus policy is a disaster. Waited 40 minutes today. Awful.",
    "Not sure how I feel about the new transport plan yet. We will see.",
    "Great to see the city investing in public transport. Long overdue!",
    "The bus policy is terrible and expensive. Who approved this??",
    "Cycling to work now instead of the bus. The new plan is useless.",
    "Honestly the new buses are clean and fast. I am impressed!",
    "More buses, fewer cars. This is exactly what we needed.",
]

print(f"Our corpus has {len(corpus)} documents.\n")
for i, doc in enumerate(corpus):
    print(f"D{i+1}: {doc}")

> **✏️ Exercise 2**
>
> Loop over the corpus and print the **length in characters** of each document.
> Which document is the longest?


In [ ]:
# ✅ Solution
for i, doc in enumerate(corpus):
    print(f"D{i+1}: {len(doc)} characters")

# Find the longest programmatically
longest_index = max(range(len(corpus)), key=lambda i: len(corpus[i]))
print(f"\nLongest is D{longest_index + 1} with {len(corpus[longest_index])} characters.")

## 3. Tokenization

In [ ]:
example = corpus[0]
print("Original:", example)
print()
print("Naive split:  ", example.split())
print()
print("NLTK tokens:  ", word_tokenize(example))

In [ ]:
tokens = word_tokenize(corpus[0].lower())
print("Tokens:", tokens)
print()
print("Number of tokens (total words):", len(tokens))
print("Number of types (distinct words):", len(set(tokens)))

> **✏️ Exercise 3**
>
> Tokenize the **second** document in the corpus (remember Python indexing starts at 0).
> How many tokens does it have? How many types?


In [ ]:
# ✅ Solution
doc2_tokens = word_tokenize(corpus[1].lower())

print("Document 2:", corpus[1])
print("Tokens:", doc2_tokens)
print()
print("Number of tokens:", len(doc2_tokens))
print("Number of types: ", len(set(doc2_tokens)))

## 4. Preprocessing: cleaning up the text

### 4.1 Lowercasing

In [ ]:
sample = "The BUS is Great but the Policy is Terrible"
print(sample.lower())

### 4.2 Removing punctuation

In [ ]:
tokens = word_tokenize("The bus policy is terrible and expensive. Who approved this??".lower())
print("Before:", tokens)

tokens_no_punct = [t for t in tokens if t.isalpha()]
print("After: ", tokens_no_punct)

### 4.3 Removing stopwords

In [ ]:
stop_words = set(stopwords.words("english"))
print("A few English stopwords:", list(stop_words)[:15])
print("Total stopwords:", len(stop_words))

In [ ]:
tokens = word_tokenize("the bus policy is terrible and expensive".lower())
tokens_no_stop = [t for t in tokens if t not in stop_words]

print("Before:", tokens)
print("After: ", tokens_no_stop)

### 4.4 Stemming vs. lemmatization

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["running", "runs", "ran", "easily", "buses", "policies", "better"]

print(f"{'word':<12}{'stem':<12}{'lemma':<12}")
print("-" * 36)
for w in words:
    print(f"{w:<12}{stemmer.stem(w):<12}{lemmatizer.lemmatize(w):<12}")

> **✏️ Exercise 4**
>
> Try stemming and lemmatizing these words: `["studies", "studying", "cars", "communication"]`.
> Which method do you find gives more sensible results here?


In [ ]:
# ✅ Solution
words = ["studies", "studying", "cars", "communication"]

print(f"{'word':<16}{'stem':<16}{'lemma':<16}")
print("-" * 48)
for w in words:
    print(f"{w:<16}{stemmer.stem(w):<16}{lemmatizer.lemmatize(w):<16}")

# Comment:
# Stemming is aggressive: "studies"/"studying" -> "studi"/"studi" (a non-word),
# and "communication" -> "commun". Lemmatization keeps real words ("study" needs
# a POS tag to work fully, but "cars" -> "car" is clean). For readability,
# lemmatization is usually nicer; for pure matching, stemming can be enough.

## 5. Putting it together: a preprocessing function

In [ ]:
def preprocess(text, remove_stopwords=True, do_lemmatize=True):
    """Clean a single document and return a list of tokens."""
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    if do_lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens


print("Original: ", corpus[0])
print("Cleaned:  ", preprocess(corpus[0]))

In [ ]:
cleaned_corpus = [preprocess(doc) for doc in corpus]

for i, tokens in enumerate(cleaned_corpus):
    print(f"D{i+1}: {tokens}")

> **✏️ Exercise 5**
>
> Run `preprocess` on the disaster tweet (document index 1) **twice**: once with
> `remove_stopwords=True` and once with `remove_stopwords=False`. What is the difference?
> Can you find a case in our corpus where removing stopwords loses important meaning?


In [ ]:
# ✅ Solution
print("With stopwords removed:   ", preprocess(corpus[1], remove_stopwords=True))
print("With stopwords kept:      ", preprocess(corpus[1], remove_stopwords=False))

# The version that keeps stopwords retains words like "is" and "a".
# Where it matters: document 2 ("Not sure how I feel...") — removing stopwords
# deletes "not", and negation is exactly what carries the meaning of a neutral/
# hedged opinion. Same risk with "fewer cars" in document 7.
print()
print("Doc 3 without stopwords:", preprocess(corpus[2], remove_stopwords=True))
print("  -> notice 'not' is gone, which changes the sentiment!")

## 6. Counting word frequencies

In [ ]:
all_tokens = []
for tokens in cleaned_corpus:
    all_tokens.extend(tokens)

print("Total tokens across the corpus:", len(all_tokens))
print("First 20:", all_tokens[:20])

In [ ]:
word_counts = Counter(all_tokens)

print("Most common words:")
for word, count in word_counts.most_common(10):
    print(f"  {word:<12} {count}")

> **✏️ Exercise 6**
>
> Use `word_counts` to answer: how many times does the word `"bus"` appear? And `"policy"`?


In [ ]:
# ✅ Solution
print("bus:   ", word_counts["bus"])
print("policy:", word_counts["policy"])

## 7. Visualizing word frequencies

In [ ]:
top_words = word_counts.most_common(10)
words = [w for w, c in top_words]
counts = [c for w, c in top_words]

plt.figure(figsize=(10, 5))
plt.bar(words, counts, color="#34B233")
plt.title("Top 10 words in the corpus")
plt.xlabel("Word")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

> **✏️ Exercise 7**
>
> Change the chart to show the top **15** words instead of 10. Then try changing the color.


In [ ]:
# ✅ Solution
top_words = word_counts.most_common(15)
words = [w for w, c in top_words]
counts = [c for w, c in top_words]

plt.figure(figsize=(11, 5))
plt.bar(words, counts, color="steelblue")   # try any color you like
plt.title("Top 15 words in the corpus")
plt.xlabel("Word")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 8. Word clouds

In [ ]:
from wordcloud import WordCloud

text_for_cloud = " ".join(all_tokens)

wc = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="Greens",
).generate(text_for_cloud)

plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Word cloud of the corpus")
plt.show()

> **✏️ Exercise 8**
>
> Generate a word cloud using only the tokens from the **positive** tweets
> (documents 0, 3, 6, 7). Do the prominent words look different from the full corpus?


In [ ]:
# ✅ Solution
positive_indices = [0, 3, 6, 7]

positive_tokens = []
for i in positive_indices:
    positive_tokens.extend(cleaned_corpus[i])

positive_text = " ".join(positive_tokens)

wc_pos = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="Greens",
).generate(positive_text)

plt.figure(figsize=(12, 6))
plt.imshow(wc_pos, interpolation="bilinear")
plt.axis("off")
plt.title("Word cloud — positive tweets only")
plt.show()

# You should see words like "love", "great", "clean", "fast", "impressed"
# stand out more than in the full-corpus cloud.

## Wrap-up

That's the full set of solutions. Remember: these are *one* way to solve each exercise,
not the only way.

### Optional challenge

Write a function `top_words_for(doc_index, n=5)` that takes a document's index and returns
its `n` most common words *after preprocessing*.


In [ ]:
# ✅ Solution
def top_words_for(doc_index, n=5):
    """Return the n most common words in a document, after preprocessing."""
    tokens = preprocess(corpus[doc_index])
    counts = Counter(tokens)
    return counts.most_common(n)


# Test on a few documents
for i in [0, 1, 4]:
    print(f"D{i+1}: {corpus[i]}")
    print(f"     Top words: {top_words_for(i)}")
    print()